In [1]:
import sys
sys.path.append("..") 

import pandas as pd
import numpy as np
from scripts.embedding_pipeline_st import EmbeddingPipeline
from sklearn.metrics.pairwise import cosine_similarity

/home/hice1/pmaji3/scratch/gnn02/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
input_dir = "../data/"

In [3]:
df = pd.read_parquet(input_dir + 'amazon_products_with_categories.parquet')
print("Data loaded successfully!")

FileNotFoundError: [Errno 2] No such file or directory: '../data/amazon_products_with_categories.parquet'

In [4]:
# Initialize Harrier
pipeline = EmbeddingPipeline(model_name='microsoft/harrier-oss-v1-270m')

# Generate embeddings
print("Computing Title-Only Embeddings...")
title_embeddings = pipeline.compute_embeddings(df, 'title')

Loading model 'microsoft/harrier-oss-v1-270m' on device: CUDA


Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'full_attention', 'sliding_attention'}
Loading weights: 100%|██████████| 236/236 [00:00<00:00, 1048.36it/s]
Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'full_attention', 'sliding_attention'}


Computing Title-Only Embeddings...


NameError: name 'df' is not defined

In [5]:
def get_similar_titles(df, embeddings, query_idx, label):
    # Prepare the Harrier-specific instruction query
    query_text = f"Instruct: Retrieve semantically similar product titles\nQuery: {df.iloc[query_idx]['title']}"
    query_vec = pipeline.model.encode([query_text])
    
    # Calculate similarities
    sims = cosine_similarity(query_vec, embeddings)[0]
    top_indices = sims.argsort()[-6:][::-1][1:] # Top 5 excluding self
    
    print(f"\n--- {label} Results ---")
    print(f"QUERY: {df.iloc[query_idx]['title']}")
    for idx in top_indices:
        print(f"[{sims[idx]:.4f}] {df.iloc[idx]['title']} | ({df.iloc[idx]['category_hierarchy'].split('|')[0][:50]}...)")

In [7]:
# Test with index
test_idx = 5
get_similar_titles(df, title_embeddings, test_idx, "TITLE-ONLY")


--- TITLE-ONLY Results ---
QUERY: How the Other Half Lives: Studies Among the Tenements of New York
[0.8072] How the Other Half Lives: Studies Among the Tenements of New York (Penguin Classics) | (Books > Subjects > Nonfiction > Current Events > P...)
[0.6793] Tenement: Immigrant Life on the Lower East Side | (Books > Subjects > Children's Books > People & Pla...)
[0.6726] Luxury Apartment Houses of Manhattan : An Illustrated History | (Books > Subjects > Professional & Technical > Arch...)
[0.6685] A History of Housing in New York City | (Books > Subjects > Professional & Technical > Arch...)
[0.6625] Manhattan for Rent, 1785-1850 | (Books > Subjects > Business & Investing > Industri...)
